# Predictive Maintenance for Industrial Robotic Arms
## Using Machine Learning on the CWRU Bearing Dataset

**Course Project — Comparative Analysis of ML Models**

---

### Phases Covered in This Notebook
- **Phase 1**: Data Loading & Preparation
- **Phase 2A**: KNN Classifier
- **Phase 2B**: SVM Classifier
- **Phase 2C**: Decision Tree Classifier
- **Phase 2D**: Polynomial Regression (Severity Prediction)
- **Phase 2E**: K-Means Clustering
- **Phase 2F**: Q-Learning (Reinforcement Learning)
- **Phase 3**: Ensemble Learning (Random Forest, AdaBoost, Gradient Boosting)
- **Phase 4**: Comparative Analysis & Evaluation

---
## 0. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, time, os

from sklearn.model_selection import (train_test_split, GridSearchCV,
                                     cross_val_score, StratifiedKFold)
from sklearn.preprocessing import StandardScaler, LabelEncoder, PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score, precision_score,
                             recall_score, mean_squared_error,
                             mean_absolute_error, r2_score,
                             silhouette_score, davies_bouldin_score)
from sklearn.ensemble import (RandomForestClassifier, AdaBoostClassifier,
                              GradientBoostingClassifier)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
warnings.filterwarnings('ignore')

# Create output directories
for d in ['plots/eda', 'plots/classifiers', 'plots/regression',
          'plots/clustering', 'plots/reinforcement_learning',
          'plots/ensemble', 'plots/comparison']:
    os.makedirs(d, exist_ok=True)

print('All libraries loaded. Output folders ready.')

---
## 1. Data Loading & Preparation

In [ ]:
df = pd.read_csv('data/feature_time_48k_2048_load_1.csv')
print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'\nFault classes ({df["fault"].nunique()}):')
print(df['fault'].value_counts())
df.head()

In [ ]:
def get_fault_type(fault_name):
    if 'Normal' in fault_name: return 'Normal'
    elif 'Ball' in fault_name: return 'Ball'
    elif 'IR' in fault_name: return 'InnerRace'
    elif 'OR' in fault_name: return 'OuterRace'
    return 'Unknown'

def get_severity(fault_name):
    if 'Normal' in fault_name: return 0.0
    elif '007' in fault_name: return 0.007
    elif '014' in fault_name: return 0.014
    elif '021' in fault_name: return 0.021
    return np.nan

df['fault_type'] = df['fault'].apply(get_fault_type)
df['severity'] = df['fault'].apply(get_severity)

print('New columns added:')
print(f'  fault_type: {df["fault_type"].unique()}')
print(f'  severity:   {sorted(df["severity"].unique())}')
print()
print(df[['fault', 'fault_type', 'severity']].drop_duplicates().sort_values('fault'))

In [ ]:
feature_cols = ['max', 'min', 'mean', 'sd', 'rms',
                'skewness', 'kurtosis', 'crest', 'form']
X = df[feature_cols].values

le = LabelEncoder()
y_class = le.fit_transform(df['fault_type'])
class_names = le.classes_
print(f'Classification classes: {list(class_names)}')

le10 = LabelEncoder()
y_10class = le10.fit_transform(df['fault'])

fault_mask = df['severity'] > 0
X_reg = df.loc[fault_mask, feature_cols].values
y_reg = df.loc[fault_mask, 'severity'].values
print(f'Regression samples (faulty only): {X_reg.shape[0]}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y_class, test_size=0.2, random_state=42, stratify=y_class)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f'\nTrain: {X_train.shape[0]} | Test: {X_test.shape[0]}')
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  {class_names[u]:12s}: {c}')

---
## Helper Function: Confusion Matrix & Results Storage

In [ ]:
def plot_confusion(y_true, y_pred, title, save_path, labels=None):
    cm = confusion_matrix(y_true, y_pred)
    if labels is None: labels = class_names
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Saved: {save_path}')

results = {}
print('Helper functions ready.')

---
# PHASE 2A — K-Nearest Neighbors (KNN)

**Hyperparameters tuned:** `n_neighbors` (1–20), `weights` (uniform/distance), `metric` (euclidean/manhattan)

In [ ]:
print('=' * 60)
print('PHASE 2A: K-NEAREST NEIGHBORS (KNN)')
print('=' * 60)

knn_param_grid = {
    'n_neighbors': list(range(1, 21)),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

knn_grid = GridSearchCV(KNeighborsClassifier(), knn_param_grid,
                        cv=5, scoring='accuracy', n_jobs=-1)
start = time.time()
knn_grid.fit(X_train_sc, y_train)
knn_time = time.time() - start

print(f'Best Parameters: {knn_grid.best_params_}')
print(f'Best CV Accuracy: {knn_grid.best_score_:.4f}')
print(f'Training Time: {knn_time:.2f}s')

knn_best = knn_grid.best_estimator_
y_pred_knn = knn_best.predict(X_test_sc)
knn_acc = accuracy_score(y_test, y_pred_knn)
knn_f1 = f1_score(y_test, y_pred_knn, average='weighted')

print(f'\nTest Accuracy: {knn_acc:.4f}')
print(f'Test F1-Score: {knn_f1:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_knn, target_names=class_names))

results['KNN'] = {'accuracy': knn_acc, 'f1': knn_f1,
    'precision': precision_score(y_test, y_pred_knn, average='weighted'),
    'recall': recall_score(y_test, y_pred_knn, average='weighted'),
    'train_time': knn_time, 'cv_score': knn_grid.best_score_}

In [ ]:
best_w = knn_grid.best_params_['weights']
best_m = knn_grid.best_params_['metric']
k_range = range(1, 21)
k_scores = [cross_val_score(KNeighborsClassifier(n_neighbors=k, weights=best_w, metric=best_m),
             X_train_sc, y_train, cv=5, scoring='accuracy').mean() for k in k_range]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(k_range, k_scores, 'bo-', linewidth=2, markersize=6)
ax.axvline(x=knn_grid.best_params_['n_neighbors'], color='red', linestyle='--',
           label=f"Best K = {knn_grid.best_params_['n_neighbors']}")
ax.set_xlabel('Number of Neighbors (K)'); ax.set_ylabel('CV Accuracy')
ax.set_title('KNN — Accuracy vs. K Value'); ax.legend(); ax.set_xticks(list(k_range))
plt.tight_layout()
plt.savefig('plots/classifiers/knn_accuracy_vs_k.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
plot_confusion(y_test, y_pred_knn, 'KNN Confusion Matrix', 'plots/classifiers/knn_confusion_matrix.png')

---
# PHASE 2B — Support Vector Machine (SVM)

**Hyperparameters tuned:** `C` (0.1–100), `gamma` (scale/auto/0.01–1), `kernel` (RBF)

In [ ]:
print('=' * 60)
print('PHASE 2B: SUPPORT VECTOR MACHINE (SVM)')
print('=' * 60)

svm_param_grid = {'C': [0.1, 1, 10, 100],
                  'gamma': ['scale', 'auto', 0.01, 0.1, 1], 'kernel': ['rbf']}

svm_grid = GridSearchCV(SVC(random_state=42), svm_param_grid,
                        cv=5, scoring='accuracy', n_jobs=-1)
start = time.time()
svm_grid.fit(X_train_sc, y_train)
svm_time = time.time() - start

print(f'Best Parameters: {svm_grid.best_params_}')
print(f'Best CV Accuracy: {svm_grid.best_score_:.4f}')
print(f'Training Time: {svm_time:.2f}s')

svm_best = svm_grid.best_estimator_
y_pred_svm = svm_best.predict(X_test_sc)
svm_acc = accuracy_score(y_test, y_pred_svm)
svm_f1 = f1_score(y_test, y_pred_svm, average='weighted')

print(f'\nTest Accuracy: {svm_acc:.4f}')
print(f'Test F1-Score: {svm_f1:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_svm, target_names=class_names))

results['SVM'] = {'accuracy': svm_acc, 'f1': svm_f1,
    'precision': precision_score(y_test, y_pred_svm, average='weighted'),
    'recall': recall_score(y_test, y_pred_svm, average='weighted'),
    'train_time': svm_time, 'cv_score': svm_grid.best_score_}

In [ ]:
plot_confusion(y_test, y_pred_svm, 'SVM Confusion Matrix', 'plots/classifiers/svm_confusion_matrix.png')

---
# PHASE 2C — Decision Tree Classifier

**Hyperparameters tuned:** `max_depth`, `min_samples_split`, `min_samples_leaf`, `criterion` (gini/entropy)

In [ ]:
print('=' * 60)
print('PHASE 2C: DECISION TREE CLASSIFIER')
print('=' * 60)

dt_param_grid = {'max_depth': [3, 5, 7, 10, 15, None],
                 'min_samples_split': [2, 5, 10, 20],
                 'min_samples_leaf': [1, 2, 5, 10],
                 'criterion': ['gini', 'entropy']}

dt_grid = GridSearchCV(DecisionTreeClassifier(random_state=42), dt_param_grid,
                       cv=5, scoring='accuracy', n_jobs=-1)
start = time.time()
dt_grid.fit(X_train_sc, y_train)
dt_time = time.time() - start

print(f'Best Parameters: {dt_grid.best_params_}')
print(f'Best CV Accuracy: {dt_grid.best_score_:.4f}')
print(f'Training Time: {dt_time:.2f}s')

dt_best = dt_grid.best_estimator_
y_pred_dt = dt_best.predict(X_test_sc)
dt_acc = accuracy_score(y_test, y_pred_dt)
dt_f1 = f1_score(y_test, y_pred_dt, average='weighted')

print(f'\nTest Accuracy: {dt_acc:.4f}')
print(f'Test F1-Score: {dt_f1:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_dt, target_names=class_names))

results['Decision Tree'] = {'accuracy': dt_acc, 'f1': dt_f1,
    'precision': precision_score(y_test, y_pred_dt, average='weighted'),
    'recall': recall_score(y_test, y_pred_dt, average='weighted'),
    'train_time': dt_time, 'cv_score': dt_grid.best_score_}

In [ ]:
importances = dt_best.feature_importances_
sorted_idx = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(len(feature_cols)), importances[sorted_idx],
       color='steelblue', edgecolor='black')
ax.set_xticks(range(len(feature_cols)))
ax.set_xticklabels([feature_cols[i] for i in sorted_idx], rotation=45)
ax.set_ylabel('Feature Importance')
ax.set_title('Decision Tree — Feature Importance Ranking')
plt.tight_layout()
plt.savefig('plots/classifiers/dt_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(24, 12))
plot_tree(dt_best, max_depth=4, feature_names=feature_cols,
          class_names=list(class_names), filled=True, rounded=True, fontsize=8, ax=ax)
ax.set_title('Decision Tree Visualization (Max 4 Levels)', fontsize=16)
plt.tight_layout()
plt.savefig('plots/classifiers/dt_tree_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
plot_confusion(y_test, y_pred_dt, 'Decision Tree Confusion Matrix', 'plots/classifiers/decision_tree_confusion_matrix.png')

---
# PHASE 2D — Polynomial Regression (Severity Prediction)

**Goal:** Predict fault severity (0.007 / 0.014 / 0.021 inches). Compare polynomial degrees 1–4.

In [ ]:
print('=' * 60)
print('PHASE 2D: POLYNOMIAL REGRESSION')
print('=' * 60)

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42)

scaler_reg = StandardScaler()
X_reg_train_sc = scaler_reg.fit_transform(X_reg_train)
X_reg_test_sc = scaler_reg.transform(X_reg_test)

reg_results = {}
for degree in range(1, 5):
    pipe = Pipeline([('poly', PolynomialFeatures(degree=degree, include_bias=False)),
                     ('reg', LinearRegression())])
    start = time.time()
    pipe.fit(X_reg_train_sc, y_reg_train)
    reg_time = time.time() - start
    y_pred_r = pipe.predict(X_reg_test_sc)
    rmse = np.sqrt(mean_squared_error(y_reg_test, y_pred_r))
    mae = mean_absolute_error(y_reg_test, y_pred_r)
    r2 = r2_score(y_reg_test, y_pred_r)
    reg_results[degree] = {'rmse': rmse, 'mae': mae, 'r2': r2,
                           'time': reg_time, 'predictions': y_pred_r}
    print(f'Degree {degree}: RMSE={rmse:.6f}, MAE={mae:.6f}, R²={r2:.4f}')

best_degree = min(reg_results, key=lambda d: reg_results[d]['rmse'])
print(f'\n>>> Best Degree: {best_degree}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
y_pred_best = reg_results[best_degree]['predictions']
axes[0].scatter(y_reg_test, y_pred_best, alpha=0.5, edgecolors='k', linewidths=0.5)
mn, mx = min(y_reg_test.min(), y_pred_best.min()), max(y_reg_test.max(), y_pred_best.max())
axes[0].plot([mn, mx], [mn, mx], 'r--', linewidth=2, label='Perfect')
axes[0].set_xlabel('Actual Severity'); axes[0].set_ylabel('Predicted')
axes[0].set_title(f'Predicted vs Actual (Degree {best_degree})'); axes[0].legend()

residuals = y_reg_test - y_pred_best
axes[1].scatter(y_pred_best, residuals, alpha=0.5, edgecolors='k', linewidths=0.5)
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Residual'); axes[1].set_title('Residual Plot')

plt.suptitle('Polynomial Regression — Severity Prediction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/regression/regression_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 5))
degrees = list(reg_results.keys())
rmses = [reg_results[d]['rmse'] for d in degrees]
r2s = [reg_results[d]['r2'] for d in degrees]
ax1.bar([d-0.15 for d in degrees], rmses, 0.3, color='tab:blue', alpha=0.8, label='RMSE')
ax1.set_xlabel('Polynomial Degree'); ax1.set_ylabel('RMSE', color='tab:blue')
ax2 = ax1.twinx()
ax2.bar([d+0.15 for d in degrees], r2s, 0.3, color='tab:orange', alpha=0.8, label='R²')
ax2.set_ylabel('R²', color='tab:orange')
ax1.set_title('Regression Performance vs Polynomial Degree'); ax1.set_xticks(degrees)
fig.legend(loc='upper right', bbox_to_anchor=(0.88, 0.88))
plt.tight_layout()
plt.savefig('plots/regression/regression_degree_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
# PHASE 2E — K-Means Clustering

**Goal:** Discover natural groupings without labels. Elbow, silhouette, Davies-Bouldin, PCA visualization.

In [ ]:
print('=' * 60)
print('PHASE 2E: K-MEANS CLUSTERING')
print('=' * 60)

X_all_sc = StandardScaler().fit_transform(X)
K_range = range(2, 16)
inertias, sil_scores, db_scores = [], [], []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    km.fit(X_all_sc)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_all_sc, km.labels_))
    db_scores.append(davies_bouldin_score(X_all_sc, km.labels_))
    print(f'K={k:2d}: Inertia={km.inertia_:10.1f}, Sil={sil_scores[-1]:.4f}, DB={db_scores[-1]:.4f}')

best_k = list(K_range)[np.argmax(sil_scores)]
print(f'\n>>> Best K: {best_k} (Silhouette={max(sil_scores):.4f})')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(list(K_range), inertias, 'bo-', lw=2, ms=6)
axes[0].axvline(x=best_k, color='red', ls='--', label=f'Best K={best_k}')
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia'); axes[0].set_title('Elbow Method'); axes[0].legend()

axes[1].plot(list(K_range), sil_scores, 'go-', lw=2, ms=6)
axes[1].axvline(x=best_k, color='red', ls='--', label=f'Best K={best_k}')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette'); axes[1].set_title('Silhouette Analysis'); axes[1].legend()

axes[2].plot(list(K_range), db_scores, 'ro-', lw=2, ms=6)
axes[2].axvline(x=best_k, color='blue', ls='--', label=f'Best K={best_k}')
axes[2].set_xlabel('K'); axes[2].set_ylabel('DB Index'); axes[2].set_title('Davies-Bouldin (lower=better)'); axes[2].legend()

plt.suptitle('K-Means — Cluster Quality', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/clustering/kmeans_elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = km_final.fit_predict(X_all_sc)
km_4 = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_labels_4 = km_4.fit_predict(X_all_sc)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_all_sc)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for i, name in enumerate(class_names):
    mask = y_class == i
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], label=name, alpha=0.5, s=20)
axes[0].set_title('True Fault Types'); axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2'); axes[0].legend()

for i in range(4):
    mask = cluster_labels_4 == i
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], label=f'Cluster {i}', alpha=0.5, s=20)
axes[1].set_title('K-Means (K=4)'); axes[1].set_xlabel('PC1'); axes[1].legend()

for i in range(best_k):
    mask = cluster_labels == i
    axes[2].scatter(X_pca[mask, 0], X_pca[mask, 1], label=f'Cluster {i}', alpha=0.5, s=20)
axes[2].set_title(f'K-Means (K={best_k})'); axes[2].set_xlabel('PC1'); axes[2].legend()

plt.suptitle('K-Means — PCA 2D Projection', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/clustering/kmeans_pca_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'K=4:  Silhouette={silhouette_score(X_all_sc, cluster_labels_4):.4f}, DB={davies_bouldin_score(X_all_sc, cluster_labels_4):.4f}')
print(f'K={best_k}: Silhouette={silhouette_score(X_all_sc, cluster_labels):.4f}, DB={davies_bouldin_score(X_all_sc, cluster_labels):.4f}')

---
# PHASE 2F — Q-Learning (Reinforcement Learning)

**States:** Normal, Low, Medium, High | **Actions:** Continue, Schedule, Emergency Stop

In [ ]:
print('=' * 60)
print('PHASE 2F: Q-LEARNING')
print('=' * 60)

n_states, n_actions = 4, 3
state_names = ['Normal', 'Low Fault', 'Medium Fault', 'High Fault']
action_names = ['Continue', 'Schedule Maint.', 'Emergency Stop']

reward_matrix = np.array([[10, -5, -20], [-5, 15, -10], [-20, 10, 5], [-50, 0, 20]])

def get_next_state(state, action):
    if action == 2: return 0
    elif action == 1: return 0 if np.random.random() < 0.8 else max(0, state - 1)
    else:
        if state == 0: return 1 if np.random.random() < 0.1 else 0
        r = np.random.random()
        if r < 0.4: return min(3, state + 1)
        elif r < 0.8: return state
        return max(0, state - 1)

Q = np.zeros((n_states, n_actions))
episode_rewards = []
epsilon = 1.0

for ep in range(10000):
    state = np.random.randint(0, n_states)
    total_reward = 0
    for _ in range(50):
        action = np.random.randint(0, n_actions) if np.random.random() < epsilon else np.argmax(Q[state])
        reward = reward_matrix[state, action]
        ns = get_next_state(state, action)
        Q[state, action] += 0.1 * (reward + 0.95 * np.max(Q[ns]) - Q[state, action])
        total_reward += reward
        state = ns
    episode_rewards.append(total_reward)
    epsilon = max(0.01, epsilon * 0.999)

print(f'Training complete! Final avg reward: {np.mean(episode_rewards[-100:]):.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
smoothed = pd.Series(episode_rewards).rolling(window=100).mean()
axes[0].plot(smoothed, color='steelblue', lw=1)
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Reward (rolling 100)')
axes[0].set_title('Q-Learning — Reward Convergence')
axes[0].axhline(y=np.mean(episode_rewards[-100:]), color='red', ls='--',
               label=f'Final: {np.mean(episode_rewards[-100:]):.1f}')
axes[0].legend()

sns.heatmap(Q, annot=True, fmt='.1f', cmap='RdYlGn',
            xticklabels=action_names, yticklabels=state_names,
            ax=axes[1], linewidths=1, linecolor='black')
axes[1].set_title('Learned Q-Table'); axes[1].set_xlabel('Action'); axes[1].set_ylabel('State')

plt.suptitle('Q-Learning Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/reinforcement_learning/qlearning_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('\nLEARNED OPTIMAL POLICY:')
print(f'{"State":<18} {"Best Action":<20} {"Q-Value":<10}')
print('-' * 50)
for s in range(n_states):
    a = np.argmax(Q[s])
    print(f'{state_names[s]:<18} {action_names[a]:<20} {Q[s, a]:.2f}')

---
# PHASE 3 — Ensemble Learning

**Random Forest** (Bagging), **AdaBoost** (Boosting), **Gradient Boosting**

In [ ]:
print('=' * 60)
print('PHASE 3A: RANDOM FOREST')
print('=' * 60)

rf_grid = GridSearchCV(RandomForestClassifier(random_state=42),
    {'n_estimators': [50, 100, 200], 'max_depth': [5, 10, 15, None],
     'min_samples_split': [2, 5, 10], 'max_features': ['sqrt', 'log2']},
    cv=5, scoring='accuracy', n_jobs=-1)

start = time.time(); rf_grid.fit(X_train_sc, y_train); rf_time = time.time() - start
rf_best = rf_grid.best_estimator_; y_pred_rf = rf_best.predict(X_test_sc)
rf_acc = accuracy_score(y_test, y_pred_rf); rf_f1 = f1_score(y_test, y_pred_rf, average='weighted')

print(f'Best: {rf_grid.best_params_}')
print(f'Accuracy: {rf_acc:.4f}, F1: {rf_f1:.4f}, Time: {rf_time:.2f}s')
print(classification_report(y_test, y_pred_rf, target_names=class_names))

results['Random Forest'] = {'accuracy': rf_acc, 'f1': rf_f1,
    'precision': precision_score(y_test, y_pred_rf, average='weighted'),
    'recall': recall_score(y_test, y_pred_rf, average='weighted'),
    'train_time': rf_time, 'cv_score': rf_grid.best_score_}

In [ ]:
plot_confusion(y_test, y_pred_rf, 'Random Forest Confusion Matrix', 'plots/ensemble/random_forest_confusion_matrix.png')

In [ ]:
print('=' * 60)
print('PHASE 3B: ADABOOST')
print('=' * 60)

ada_grid = GridSearchCV(
    AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=2),
                       random_state=42, algorithm='SAMME'),
    {'n_estimators': [50, 100, 200, 300], 'learning_rate': [0.01, 0.05, 0.1, 0.5, 1.0]},
    cv=5, scoring='accuracy', n_jobs=-1)

start = time.time(); ada_grid.fit(X_train_sc, y_train); ada_time = time.time() - start
ada_best = ada_grid.best_estimator_; y_pred_ada = ada_best.predict(X_test_sc)
ada_acc = accuracy_score(y_test, y_pred_ada); ada_f1 = f1_score(y_test, y_pred_ada, average='weighted')

print(f'Best: {ada_grid.best_params_}')
print(f'Accuracy: {ada_acc:.4f}, F1: {ada_f1:.4f}, Time: {ada_time:.2f}s')
print(classification_report(y_test, y_pred_ada, target_names=class_names))

results['AdaBoost'] = {'accuracy': ada_acc, 'f1': ada_f1,
    'precision': precision_score(y_test, y_pred_ada, average='weighted'),
    'recall': recall_score(y_test, y_pred_ada, average='weighted'),
    'train_time': ada_time, 'cv_score': ada_grid.best_score_}

In [ ]:
print('=' * 60)
print('PHASE 3C: GRADIENT BOOSTING')
print('=' * 60)

gb_grid = GridSearchCV(GradientBoostingClassifier(random_state=42),
    {'n_estimators': [100, 200], 'learning_rate': [0.05, 0.1, 0.2], 'max_depth': [3, 5, 7]},
    cv=5, scoring='accuracy', n_jobs=-1)

start = time.time(); gb_grid.fit(X_train_sc, y_train); gb_time = time.time() - start
gb_best = gb_grid.best_estimator_; y_pred_gb = gb_best.predict(X_test_sc)
gb_acc = accuracy_score(y_test, y_pred_gb); gb_f1 = f1_score(y_test, y_pred_gb, average='weighted')

print(f'Best: {gb_grid.best_params_}')
print(f'Accuracy: {gb_acc:.4f}, F1: {gb_f1:.4f}, Time: {gb_time:.2f}s')
print(classification_report(y_test, y_pred_gb, target_names=class_names))

results['Gradient Boosting'] = {'accuracy': gb_acc, 'f1': gb_f1,
    'precision': precision_score(y_test, y_pred_gb, average='weighted'),
    'recall': recall_score(y_test, y_pred_gb, average='weighted'),
    'train_time': gb_time, 'cv_score': gb_grid.best_score_}

In [ ]:
plot_confusion(y_test, y_pred_gb, 'Gradient Boosting Confusion Matrix', 'plots/ensemble/gradient_boosting_confusion_matrix.png')

---
# PHASE 4 — Comparative Analysis & Evaluation

In [ ]:
print('=' * 60)
print('PHASE 4: COMPARATIVE ANALYSIS')
print('=' * 60)

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
models = {'KNN': knn_best, 'SVM': svm_best, 'Decision Tree': dt_best,
          'Random Forest': rf_best, 'AdaBoost': ada_best, 'Gradient Boosting': gb_best}

cv_results = {}
print(f'{"Model":<22} {"Mean CV Acc":<14} {"Std":<10}')
print('-' * 46)
for name, model in models.items():
    scores = cross_val_score(model, X_train_sc, y_train, cv=cv, scoring='accuracy')
    cv_results[name] = scores
    print(f'{name:<22} {scores.mean():.4f}        +/-{scores.std():.4f}')

In [ ]:
comparison_data = []
for mn, res in results.items():
    comparison_data.append({'Model': mn, 'Accuracy': f"{res['accuracy']:.4f}",
        'Precision': f"{res['precision']:.4f}", 'Recall': f"{res['recall']:.4f}",
        'F1-Score': f"{res['f1']:.4f}", 'CV Score': f"{res['cv_score']:.4f}",
        'Train Time': f"{res['train_time']:.2f}s"})

comparison_df = pd.DataFrame(comparison_data).sort_values('Accuracy', ascending=False)
print(comparison_df.to_string(index=False))
comparison_df

In [ ]:
model_names = list(results.keys())
accuracies = [results[m]['accuracy'] for m in model_names]
f1_vals = [results[m]['f1'] for m in model_names]
x = np.arange(len(model_names)); width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
b1 = ax.bar(x - width/2, accuracies, width, label='Accuracy', color='steelblue', edgecolor='black')
b2 = ax.bar(x + width/2, f1_vals, width, label='F1-Score', color='coral', edgecolor='black')
for b in b1: ax.text(b.get_x()+b.get_width()/2., b.get_height()+0.005, f'{b.get_height():.3f}', ha='center', fontsize=9)
for b in b2: ax.text(b.get_x()+b.get_width()/2., b.get_height()+0.005, f'{b.get_height():.3f}', ha='center', fontsize=9)
ax.set_xlabel('Model'); ax.set_ylabel('Score'); ax.set_title('Model Comparison — Accuracy & F1')
ax.set_xticks(x); ax.set_xticklabels(model_names, rotation=30, ha='right')
ax.legend(); ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.savefig('plots/comparison/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
bp = ax.boxplot([cv_results[m] for m in model_names], labels=model_names, patch_artist=True, notch=True)
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']
for patch, c in zip(bp['boxes'], colors): patch.set_facecolor(c); patch.set_alpha(0.7)
ax.set_ylabel('Accuracy'); ax.set_title('10-Fold CV — Accuracy Distribution')
plt.xticks(rotation=30, ha='right'); plt.tight_layout()
plt.savefig('plots/comparison/cv_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
train_times = [results[m]['train_time'] for m in model_names]
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(model_names, train_times, color=colors, edgecolor='black')
for i, t in enumerate(train_times): ax.text(t + 0.1, i, f'{t:.2f}s', va='center')
ax.set_xlabel('Training Time (s)'); ax.set_title('Training Time Comparison')
plt.tight_layout()
plt.savefig('plots/comparison/training_time_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('\n' + '=' * 70)
print('REGRESSION RESULTS')
print('=' * 70)
for deg, res in reg_results.items():
    m = ' <<<' if deg == best_degree else ''
    print(f'Degree {deg}: RMSE={res["rmse"]:.6f}, MAE={res["mae"]:.6f}, R²={res["r2"]:.4f}{m}')

In [ ]:
print('\n' + '=' * 70)
print('FINAL SUMMARY')
print('=' * 70)

best_model = max(results, key=lambda m: results[m]['accuracy'])
fastest = min(results, key=lambda m: results[m]['train_time'])
indiv = max(['KNN','SVM','Decision Tree'], key=lambda m: results[m]['accuracy'])
ens = max(['Random Forest','AdaBoost','Gradient Boosting'], key=lambda m: results[m]['accuracy'])
imp = results[ens]['accuracy'] - results[indiv]['accuracy']

print(f'\n1. BEST CLASSIFIER: {best_model} ({results[best_model]["accuracy"]:.4f})')
print(f'2. FASTEST MODEL: {fastest} ({results[fastest]["train_time"]:.2f}s)')
print(f'3. ENSEMBLE IMPROVEMENT: {indiv}({results[indiv]["accuracy"]:.4f}) -> {ens}({results[ens]["accuracy"]:.4f}) = {imp:+.4f}')
print(f'4. REGRESSION: Best degree={best_degree}, R²={reg_results[best_degree]["r2"]:.4f}')
print(f'5. CLUSTERING: Best K={best_k}')
print(f'6. Q-LEARNING: Converged, avg reward={np.mean(episode_rewards[-100:]):.1f}')
print(f'\nRECOMMENDATION: {best_model} for accuracy, {fastest} for speed, Decision Tree for interpretability')
print('\n' + '=' * 70)
print('ALL PHASES COMPLETE!')
print('=' * 70)